# Performance Benchmarking Analysis

Comprehensive analysis of parallel processing performance using the AIS data pipeline.

To generate the benchmark data for this analysis, run the following commands in the workspace root directory 

(or better run `python scripts\run_benchmarks.py`):

```bash
# Define parameters
declare -a WORKERS=(1 2 3 4)
declare -a CHUNK_SIZES=(10000 50000 100000 200000 500000 1000000)

INPUT_FILE="data/sample/2025-09-01_head_100_000_rows.csv"

# Run all combinations
for w in "${WORKERS[@]}"; do
  for c in "${CHUNK_SIZES[@]}"; do
    RUN_NAME="benchmark_w${w}_c$((c/1000))k"
    OUTPUT_DIR="data/output/$RUN_NAME"
    
    echo "Running: $RUN_NAME (workers=$w, chunk_size=$c)"
    mkdir -p "$OUTPUT_DIR"
    
    python -m scripts.run_detection \
      "$INPUT_FILE" \
      --chunk-size "$c" \
      --workers "$w" \
      --top 10 \
      --memory-output "$OUTPUT_DIR/memory_profile.csv" \
      --output "$OUTPUT_DIR/dfsi_results.csv"
  done
done
```

Once benchmarks complete, re-run all cells in this notebook to analyze results.
The plots will be placed in `data\output`.

Summary for the performance results is provided below.

**Expected output structure:**
```
data/output/
├── benchmark_w1_c10k/
│   ├── memory_profile.csv
│   ├── worker_memory_profile.csv
│   ├── memory_summary.csv
│   └── dfsi_results.csv
├── benchmark_w1_c50k/
│   └── ...
└── benchmark_w4_c1000k/
    └── ...
```

In [ ]:
import os
import sys
import subprocess
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from collections import defaultdict
from scipy.optimize import minimize_scalar
import warnings
warnings.filterwarnings('ignore')

# Set style for better-looking plots
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 8)
plt.rcParams['font.size'] = 10

# Add src to path
sys.path.insert(0, str(Path.cwd().parent))

# Define directories
WORKSPACE_ROOT = Path.cwd().parent
DATA_OUTPUT_DIR = WORKSPACE_ROOT / 'data' / 'output'
SAMPLE_DATA = WORKSPACE_ROOT / 'data' / 'sample' / '2025-09-01_head_100_000_rows.csv'

print('✓ Setup complete')
print(f'  Workspace: {WORKSPACE_ROOT}')
print(f'  Output dir: {DATA_OUTPUT_DIR}')
print(f'  Sample data: {SAMPLE_DATA.exists()}')

In [ ]:
# (A) MEMORY TIMELINE ANALYSIS
# Plot memory usage over time for each benchmark run
# Shows main_rss_mb, workers_rss_mb, and total_rss_mb together

benchmark_dirs = sorted(DATA_OUTPUT_DIR.glob('benchmark_*_c*'))
print(f'Found {len(benchmark_dirs)} benchmark directories')

for bench_dir in benchmark_dirs[:5]:  # Plot first 5 runs
    memory_file = bench_dir / 'memory_profile.csv'
    print(memory_file)
    if memory_file.exists():
        data = pd.read_csv(memory_file)
        # Filter for timer samples only (exclude event entries)
        data = data[data['sample_kind'] == 'timer']
        fig, ax = plt.subplots(figsize=(13, 6))
        
        # Plot all three memory components
        ax.plot(data['elapsed_time_sec'], data['main_rss_mb'], marker='o', markersize=2, label='Main RSS', linewidth=1.5)
        ax.plot(data['elapsed_time_sec'], data['workers_rss_mb'], marker='s', markersize=2, label='Workers RSS', linewidth=1.5)
        ax.plot(data['elapsed_time_sec'], data['total_rss_mb'], marker='^', markersize=2, label='Total RSS', linewidth=2, linestyle='--')
        
        ax.set_xlabel('Elapsed Time (s)', fontsize=11)
        ax.set_ylabel('Memory (MB)', fontsize=11)
        ax.set_title(f'Memory Timeline: {bench_dir.name}', fontsize=12, fontweight='bold')
        ax.legend(loc='best', fontsize=10)
        ax.grid(True, alpha=0.3)
        plt.tight_layout()
        
        # Save figure
        output_file = DATA_OUTPUT_DIR / f'{bench_dir.name}_memory_timeline.png'
        plt.savefig(output_file, dpi=150, bbox_inches='tight')
        print(f'✓ Saved: {output_file.name}')
        plt.show()

In [ ]:
# (B) PER-WORKER MEMORY ANALYSIS
# Analyze memory distribution across workers

per_worker_plotted = 0
for bench_dir in benchmark_dirs:
    worker_memory_file = bench_dir / 'worker_memory_profile.csv'
    if worker_memory_file.exists() and per_worker_plotted < 100: # change the number of plots to show max 24 combinations as for now
        worker_data = pd.read_csv(worker_memory_file)
        # Filter for timer samples only (exclude event entries)
        worker_data = worker_data[worker_data['sample_kind'] == 'timer']
        
        fig, ax = plt.subplots(figsize=(13, 6))
        for worker_idx in sorted(worker_data['worker_index'].unique()):
            w_data = worker_data[worker_data['worker_index'] == worker_idx]
            ax.plot(w_data['elapsed_time_sec'], w_data['rss_mb'], label=f'Worker {int(worker_idx)}', marker='o', markersize=2, linewidth=1.5)
        
        ax.set_xlabel('Elapsed Time (s)', fontsize=11)
        ax.set_ylabel('Memory (MB)', fontsize=11)
        ax.set_title(f'Per-Worker Memory Distribution: {bench_dir.name}', fontsize=12, fontweight='bold')
        ax.legend(loc='best', fontsize=9)
        ax.grid(True, alpha=0.3)
        plt.tight_layout()
        
        output_file = DATA_OUTPUT_DIR / f'{bench_dir.name}_per_worker_memory.png'
        plt.savefig(output_file, dpi=150, bbox_inches='tight')
        print(f'✓ Saved: {output_file.name}')
        plt.show()
        per_worker_plotted += 1

In [ ]:
# (C) SPEEDUP VS WORKERS ANALYSIS
# Calculate and plot speedup for each chunk size

# Extract metrics from all benchmark runs with all peak RSS values
metrics_list = []
for bench_dir in benchmark_dirs:
    summary_file = bench_dir / 'memory_summary.csv'
    if summary_file.exists():
        summary_data = pd.read_csv(summary_file)
        
        # Extract run parameters from directory name (e.g., benchmark_w2_c100k)
        run_name = bench_dir.name
        parts = run_name.split('_')
        workers = int(parts[1][1:])  # w2 -> 2
        chunk_k = int(parts[2][1:-1])  # c100k -> 100
        
        # Calculate metrics - extract all peak memory values
        metrics_list.append({
            'run_name': run_name,
            'workers': workers,
            'chunk_size_k': chunk_k,
            'runtime': summary_data['duration_sec'].values[0] if len(summary_data) > 0 else None,
            'peak_main_rss_mb': summary_data['peak_main_rss_mb'].values[0] if len(summary_data) > 0 else None,
            'peak_workers_rss_mb': summary_data['peak_workers_rss_mb'].values[0] if len(summary_data) > 0 else None,
            'peak_total_rss_mb': summary_data['peak_total_rss_mb'].values[0] if len(summary_data) > 0 else None,
            'peak_single_worker_rss_mb': summary_data['peak_single_worker_rss_mb'].values[0] if len(summary_data) > 0 else None,
        })

metrics_df = pd.DataFrame(metrics_list)
print(f'Extracted metrics from {len(metrics_df)} runs')
print(f'Worker counts: {sorted(metrics_df["workers"].unique())}')
print(f'Chunk sizes (K): {sorted(metrics_df["chunk_size_k"].unique())}')

if len(metrics_df) > 0:
    # Calculate speedup for each chunk size
    fig, ax = plt.subplots(figsize=(13, 7))
    speedup_data = {}
    
    for chunk_k in sorted(metrics_df['chunk_size_k'].unique()):
        chunk_metrics = metrics_df[metrics_df['chunk_size_k'] == chunk_k].sort_values('workers')
        if len(chunk_metrics) > 0 and 'runtime' in chunk_metrics.columns:
            baseline_runtime = chunk_metrics[chunk_metrics['workers'] == 1]['runtime'].values
            if len(baseline_runtime) > 0:
                baseline = baseline_runtime[0]
                speedups = baseline / chunk_metrics['runtime'].values
                chunk_label = f'{chunk_k}K chunks'
                ax.plot(chunk_metrics['workers'], speedups, marker='o', label=chunk_label, linewidth=2, markersize=8)
                speedup_data[chunk_label] = list(zip(chunk_metrics['workers'], speedups))
    
    ax.set_xlabel('Number of Workers', fontsize=11)
    ax.set_ylabel('Speedup (×)', fontsize=11)
    ax.set_title('Speedup vs Number of Workers', fontsize=12, fontweight='bold')
    ax.legend(fontsize=10, loc='best')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    
    output_file = DATA_OUTPUT_DIR / 'speedup_vs_workers.png'
    plt.savefig(output_file, dpi=150, bbox_inches='tight')
    print(f'✓ Saved: {output_file.name}')
    plt.show()

In [ ]:
# (D) CHUNK SIZE IMPACT ANALYSIS
# Analyze runtime and memory trends across chunk sizes
# Show all 4 workers as separate lines

if len(metrics_df) > 0:
    worker_counts = sorted(metrics_df['workers'].unique())
    chunk_sizes_sorted = sorted(metrics_df['chunk_size_k'].unique())
    
    # Create figure with 2 subplots: runtime and memory
    fig, (ax_runtime, ax_memory) = plt.subplots(1, 2, figsize=(16, 6))
    
    # Define colors for each worker count (dynamically generate if more than 4)
    if len(worker_counts) <= 4:
        colors_workers = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']  # Blue, Orange, Green, Red
    else:
        colors_workers = plt.cm.tab10(np.linspace(0, 1, len(worker_counts)))
    
    # Plot runtime and memory for each worker count as separate lines
    for idx, workers_count in enumerate(worker_counts):
        worker_data = metrics_df[metrics_df['workers'] == workers_count].sort_values('chunk_size_k')
        
        if len(worker_data) > 0:
            # Runtime line
            ax_runtime.plot(worker_data['chunk_size_k'], worker_data['runtime'], 
                           marker='o', linewidth=2.5, markersize=8, 
                           label=f'{int(workers_count)} workers',
                           color=colors_workers[idx], alpha=0.8)
            
            # Memory line (total only)
            ax_memory.plot(worker_data['chunk_size_k'], worker_data['peak_workers_rss_mb'], 
                          marker='s', linewidth=2.5, markersize=8,
                          label=f'{int(workers_count)} workers',
                          color=colors_workers[idx], alpha=0.8)
    
    # Runtime subplot formatting
    ax_runtime.set_xlabel('Chunk Size (K)', fontsize=12, fontweight='bold')
    ax_runtime.set_ylabel('Runtime (seconds)', fontsize=12, fontweight='bold')
    ax_runtime.set_title('Runtime vs Chunk Size', fontsize=13, fontweight='bold')
    ax_runtime.legend(fontsize=11, loc='upper right', framealpha=0.95)
    ax_runtime.grid(True, alpha=0.3)
    
    # Memory subplot formatting
    ax_memory.set_xlabel('Chunk Size (K)', fontsize=12, fontweight='bold')
    ax_memory.set_ylabel('Peak Worker Memory (MB)', fontsize=12, fontweight='bold')
    ax_memory.set_title('Peak Worker Memory vs Chunk Size', fontsize=13, fontweight='bold')
    ax_memory.legend(fontsize=11, loc='upper right', framealpha=0.95)
    ax_memory.grid(True, alpha=0.3)
    
    plt.tight_layout()
    output_file = DATA_OUTPUT_DIR / 'chunk_size_impact.png'
    plt.savefig(output_file, dpi=150, bbox_inches='tight')
    print(f'✓ Saved: {output_file.name}')
    plt.show()

In [ ]:
# (D) CHUNK SIZE IMPACT ANALYSIS — SEPARATE PLOTS

if len(metrics_df) > 0:
    worker_counts = sorted(metrics_df['workers'].unique())
    chunk_sizes_sorted = sorted(metrics_df['chunk_size_k'].unique())

    # Define colors for each worker count
    if len(worker_counts) <= 4:
        colors_workers = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']
    else:
        colors_workers = plt.cm.tab10(np.linspace(0, 1, len(worker_counts)))

    # ============================================================
    # 1) RUNTIME PLOT
    # ============================================================
    plt.figure(figsize=(10, 6))

    for idx, workers_count in enumerate(worker_counts):
        worker_data = metrics_df[metrics_df['workers'] == workers_count].sort_values('chunk_size_k')

        if len(worker_data) > 0:
            plt.plot(worker_data['chunk_size_k'], worker_data['runtime'],
                     marker='o', linewidth=2.5, markersize=8,
                     label=f'{int(workers_count)} workers',
                     color=colors_workers[idx], alpha=0.85)

    plt.xlabel('Chunk Size (K)', fontsize=12, fontweight='bold')
    plt.ylabel('Runtime (seconds)', fontsize=12, fontweight='bold')
    plt.title('Runtime vs Chunk Size', fontsize=14, fontweight='bold')
    plt.grid(True, alpha=0.3)
    plt.legend(fontsize=11, loc='upper right')

    output_file_runtime = DATA_OUTPUT_DIR / 'chunk_size_runtime.png'
    plt.savefig(output_file_runtime, dpi=150, bbox_inches='tight')
    print(f'✓ Saved: {output_file_runtime.name}')
    plt.show()

    # ============================================================
    # 2) MEMORY PLOT
    # ============================================================
    plt.figure(figsize=(10, 6))

    for idx, workers_count in enumerate(worker_counts):
        worker_data = metrics_df[metrics_df['workers'] == workers_count].sort_values('chunk_size_k')

        if len(worker_data) > 0:
            plt.plot(worker_data['chunk_size_k'], worker_data['peak_workers_rss_mb'],
                     marker='s', linewidth=2.5, markersize=8,
                     label=f'{int(workers_count)} workers',
                     color=colors_workers[idx], alpha=0.85)

    plt.xlabel('Chunk Size (K)', fontsize=12, fontweight='bold')
    plt.ylabel('Peak Worker Memory (MB)', fontsize=12, fontweight='bold')
    plt.title('Peak Worker Memory vs Chunk Size', fontsize=14, fontweight='bold')
    plt.grid(True, alpha=0.3)
    plt.legend(fontsize=11, loc='upper right')

    output_file_memory = DATA_OUTPUT_DIR / 'chunk_size_memory.png'
    plt.savefig(output_file_memory, dpi=150, bbox_inches='tight')
    print(f'✓ Saved: {output_file_memory.name}')
    plt.show()


In [ ]:


def plot_amdahl_for_chunk(metrics_df, chunk_size_k, output_dir=None):
    """
    Plot measured vs theoretical Amdahl's Law speedup for a selected chunk size.
    Produces a single clean plot with appropriate Y-axis scaling.
    Saves PNG if output_dir is provided.
    """

    # Filter metrics for the chosen chunk size
    df = metrics_df[metrics_df['chunk_size_k'] == chunk_size_k].sort_values('workers')

    if df.empty:
        print(f"No data found for chunk size {chunk_size_k}K")
        return

    workers = df['workers'].values
    runtimes = df['runtime'].values

    # Baseline runtime (workers == 1)
    baseline_idx = np.where(workers == 1)[0]
    if len(baseline_idx) == 0:
        print("No baseline (workers=1) found; cannot compute speedup.")
        return

    baseline = runtimes[baseline_idx[0]]
    measured_speedup = baseline / runtimes

    # Amdahl's Law model
    def amdahl(w, P):
        return 1 / ((1 - P) + P / w)

    # Fit P (parallel fraction)
    def residual(P):
        return np.sum((measured_speedup - amdahl(workers, P))**2)

    result = minimize_scalar(residual, bounds=(0, 1), method='bounded')
    P = result.x

    # Generate smooth theoretical curve
    ideal_workers = np.linspace(1, workers.max(), 200)
    theoretical_speedup = amdahl(ideal_workers, P)

    # Perfect linear speedup
    ideal_linear = ideal_workers

    # --- Plot ---
    plt.figure(figsize=(10, 6))

    plt.plot(ideal_workers, ideal_linear, ':', color='gray', linewidth=2,
             label='Ideal Linear Speedup')

    plt.plot(ideal_workers, theoretical_speedup, '--', color='green', linewidth=2.5,
             label=f'Amdahl Model (P={P:.1%})')

    plt.plot(workers, measured_speedup, 'o-', color='darkblue', linewidth=2.5,
             markersize=9, label='Measured Speedup')

    plt.title(f"Amdahl's Law Analysis — Chunk Size {chunk_size_k}K", fontsize=14, fontweight='bold')
    plt.xlabel("Number of Workers", fontsize=12)
    plt.ylabel("Speedup (×)", fontsize=12)

    # Y-axis scaling appropriate for Amdahl analysis
    ymax = max(measured_speedup.max(), theoretical_speedup.max(), ideal_linear.max())
    plt.ylim(0.8, ymax * 1.15)

    plt.grid(True, alpha=0.3)
    plt.legend(fontsize=11)
    plt.tight_layout()

    # Save PNG if directory provided
    if output_dir is not None:
        output_path = output_dir / f"amdahl.png"
        plt.savefig(output_path, dpi=150, bbox_inches='tight')
        print(f"✓ Saved: {output_path.name}")

    plt.show()

    print(f"Estimated parallel fraction P = {P*100:.2f}%")
    print(f"Estimated serial fraction = {(1-P)*100:.2f}%")


In [ ]:
plot_amdahl_for_chunk(metrics_df, chunk_size_k=250, output_dir=DATA_OUTPUT_DIR)


In [ ]:
# (G) MEMORY BY WORKERS COMPARISON
# Heatmap using peak process memory = max(main, single worker)

if len(metrics_df) > 0:
    # Prepare sorted axes
    workers_sorted = sorted(metrics_df['workers'].unique())
    chunks_sorted = sorted(metrics_df['chunk_size_k'].unique())

    # Compute peak process memory for each row
    metrics_df['peak_process_rss_mb'] = metrics_df[['peak_main_rss_mb',
                                                    'peak_single_worker_rss_mb']].max(axis=1)

    # Build heatmap matrix
    process_memory_heatmap = np.zeros((len(chunks_sorted), len(workers_sorted)))

    for i, chunk_k in enumerate(chunks_sorted):
        for j, w in enumerate(workers_sorted):
            data = metrics_df[(metrics_df['chunk_size_k'] == chunk_k) &
                              (metrics_df['workers'] == w)]
            if len(data) > 0:
                process_memory_heatmap[i, j] = data['peak_process_rss_mb'].values[0]

    # Build runtime heatmap
    runtime_heatmap = np.zeros((len(chunks_sorted), len(workers_sorted)))
    for i, chunk_k in enumerate(chunks_sorted):
        for j, w in enumerate(workers_sorted):
            data = metrics_df[(metrics_df['chunk_size_k'] == chunk_k) &
                              (metrics_df['workers'] == w)]
            if len(data) > 0:
                runtime_heatmap[i, j] = data['runtime'].values[0]

    # ============================================================
    # PLOT FIGURE WITH TWO HEATMAPS
    # ============================================================
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    # ------------------------------------------------------------
    # Heatmap 1: Peak Process Memory
    # ------------------------------------------------------------
    im1 = axes[0].imshow(process_memory_heatmap, cmap='YlOrRd',
                         aspect='auto', interpolation='nearest')

    axes[0].set_xticks(range(len(workers_sorted)))
    axes[0].set_yticks(range(len(chunks_sorted)))
    axes[0].set_xticklabels([f'{int(w)} workers' for w in workers_sorted], fontsize=11, fontweight='bold')
    axes[0].set_yticklabels([f'{int(c)}K' for c in chunks_sorted], fontsize=11, fontweight='bold')

    axes[0].set_xlabel('Worker Configuration', fontsize=12, fontweight='bold')
    axes[0].set_ylabel('Chunk Size', fontsize=12, fontweight='bold')
    axes[0].set_title('Peak Process Memory (MB)', fontsize=13, fontweight='bold')

    # Add values
    for i in range(len(chunks_sorted)):
        for j in range(len(workers_sorted)):
            axes[0].text(j, i, f'{process_memory_heatmap[i, j]:.0f}',
                         ha="center", va="center", color="black",
                         fontsize=9, fontweight='bold')

    cbar1 = plt.colorbar(im1, ax=axes[0])
    cbar1.set_label('Memory (MB)', fontsize=11, fontweight='bold')

    # ------------------------------------------------------------
    # Heatmap 2: Runtime
    # ------------------------------------------------------------
    im2 = axes[1].imshow(runtime_heatmap, cmap='RdYlGn_r',
                         aspect='auto', interpolation='nearest')

    axes[1].set_xticks(range(len(workers_sorted)))
    axes[1].set_yticks(range(len(chunks_sorted)))
    axes[1].set_xticklabels([f'{int(w)} workers' for w in workers_sorted], fontsize=11, fontweight='bold')
    axes[1].set_yticklabels([f'{int(c)}K' for c in chunks_sorted], fontsize=11, fontweight='bold')

    axes[1].set_xlabel('Worker Configuration', fontsize=12, fontweight='bold')
    axes[1].set_ylabel('Chunk Size', fontsize=12, fontweight='bold')
    axes[1].set_title('Runtime (seconds)', fontsize=13, fontweight='bold')

    # Add values
    for i in range(len(chunks_sorted)):
        for j in range(len(workers_sorted)):
            axes[1].text(j, i, f'{runtime_heatmap[i, j]:.0f}',
                         ha="center", va="center", color="black",
                         fontsize=9, fontweight='bold')

    cbar2 = plt.colorbar(im2, ax=axes[1])
    cbar2.set_label('Time (sec)', fontsize=11, fontweight='bold')

    plt.tight_layout()
    output_file = DATA_OUTPUT_DIR / 'memory_by_workers_comparison.png'
    plt.savefig(output_file, dpi=150, bbox_inches='tight')
    print(f'✓ Saved: {output_file.name}')
    plt.show()


# summary

| Visualisation | Text |
|--------|-------|
| Runtime vs chunksize | The runtime versus chunk size graph shows that increasing the number of workers sharply reduces runtime from about 400 seconds with one worker to roughly 220 seconds with two or more, and this confirms that parallelization accelerates vessel anomaly analysis, while chunk size itself has only a minor influence on runtime once parallelism is active. |
| Heatmap image | The heatmaps comparing peak process memory and runtime show that memory usage scales almost linearly with chunk size, while runtime improves dramatically when moving from one to two workers and then stabilizes, froom this graph it is also seen that the 250 K chunk size offers the best balance between memory efficiency and computational throughput for vessel anomaly detection framework. |
| Speedup | The speedup versus number of workers graph demonstrates near‑linear scaling up to two workers, reaching about 1.8 times speedup, after which performance plateaus, this means that memory bandwidth and cache contention limit further gains beyond two workers. The best results are obtained with 250k chunk size as can be seen. |
| Peak worker memory versus chunk size | The peak worker memory versus chunk size graph shows that memory usage rises steadily with larger chunks and more workers, this confirms that larger chunks make each worker hold more data in memory at once, so their memory use goes up or in other words higher data granularity increases per‑process memory demand, though the growth remains predictable and manageable for the chosen 250 K configuration. |
| Per worker memory w4, 250 chunk, 100 chunk | The per‑worker memory distribution graph reveal that all workers maintain similar memory footprints during execution, fluctuating between roughly 150 and 270 MB depending on chunk size, with occasional drops which shows the task completion stages and new calculation process. |
| Amdahl’s Law | The Amdahl’s Law graph indicates that about 65 % of the workload is parallelizable, while the remaining 35 % consists of sequential merge logic and coordination, explaining the observed saturation in speedup beyond two workers. |


# hardware specifications

CPU: Intel core i7 12700H (14 cores, 24MB Cache, up to 4.70 GHz)

RAM: 16GB DDR5 4800MHz

SSD: 1TB PCIe® 4.0 NVMe™ M.2